# 01 - Análise Exploratória dos Dados (EDA)

**Projeto:** Previsão de Evasão Escolar (SCC0630)

Objetivo desta análise: entender o dataset antes de modelar — qualidade dos dados, distribuição do alvo, tipos de variáveis, relações com a evasão e implicações para a modelagem.

> Antes de rodar: `python scripts/download_data.py` (gera `data/raw/dataset.csv`).

In [ ]:
import sys
from pathlib import Path

# Permite importar o pacote src a partir do notebook
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_raw, TARGET

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
PALETTE = {'Dropout': '#d62728', 'Enrolled': '#ff7f0e', 'Graduate': '#2ca02c'}

In [ ]:
df = load_raw()
print('Dimensões:', df.shape)
df.head()

## 1. Visão geral e qualidade dos dados

4424 alunos × 36 atributos + alvo. Esperamos: ausência de nulos e de duplicatas (dataset já curado pela UCI).

In [ ]:
print('Valores nulos (total):', df.isna().sum().sum())
print('Linhas duplicadas    :', df.duplicated().sum())
print('\nTipos de dados:')
print(df.dtypes.value_counts())
df.info()

### Grupos de variáveis

Cuidado importante: a maioria dos atributos é `int64`, mas **nem todo inteiro é numérico**. Muitos são *códigos de categorias* (ex.: `Course`, `Marital Status`). Tratá-los como números em correlações/escalonamento é enganoso. Separamos em três grupos:

- **Binárias** (0/1): flags como `Debtor`, `Scholarship holder`, `Gender`.
- **Nominais codificadas**: categorias sem ordem, guardadas como inteiros (alta cardinalidade).
- **Numéricas/ordinais**: notas, idade, contagens de disciplinas e indicadores macroeconômicos.

In [ ]:
binarias = ['Daytime/evening attendance', 'Displaced', 'Educational special needs', 'Debtor',
            'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']

nominais = ['Marital Status', 'Application mode', 'Course', 'Previous qualification', 'Nacionality',
            "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]

curriculares = [c for c in df.columns if c.startswith('Curricular')]
macro = ['Unemployment rate', 'Inflation rate', 'GDP']
outras_num = ['Application order', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment']
continuas = outras_num + curriculares + macro

print(f'Binárias: {len(binarias)} | Nominais: {len(nominais)} | Numéricas: {len(continuas)}')
print('\nCardinalidade das nominais (nº de categorias):')
df[nominais].nunique().sort_values(ascending=False)

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df[continuas].describe().T.round(2)

## 2. A variável alvo

Problema multiclasse com **3 classes desbalanceadas**. Isso justifica usar F1/recall *macro* (e não só acurácia) na avaliação, e considerar estratégias para o desbalanceamento na modelagem.

In [ ]:
dist = df[TARGET].value_counts()
print(dist)
print('\nProporção:')
print((dist / len(df) * 100).round(1))

fig, ax = plt.subplots(figsize=(6, 4))
order = ['Dropout', 'Enrolled', 'Graduate']
sns.countplot(data=df, x=TARGET, order=order, palette=PALETTE, ax=ax)
ax.set_title('Distribuição do alvo')
ax.set_xlabel('')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.0f}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.show()

## 3. Análise univariada — variáveis numéricas

Distribuições das principais contínuas. Pontos de atenção: `Age at enrollment` é assimétrica à direita (maioria 18-25, com cauda longa); várias contagens de disciplinas têm muitos zeros (alunos que não fizeram nenhuma avaliação).

In [ ]:
destaque = ['Age at enrollment', 'Admission grade', 'Previous qualification (grade)',
            'Curricular units 1st sem (approved)', 'Curricular units 2nd sem (approved)',
            'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
            'Unemployment rate', 'GDP']

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.ravel(), destaque):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 4. Análise bivariada — variáveis binárias vs. evasão

Aqui aparecem os sinais mais fortes. Calculamos a **taxa de Dropout** dentro de cada grupo.

In [ ]:
def taxa_dropout(col):
    return df.groupby(col)[TARGET].apply(lambda s: (s == 'Dropout').mean() * 100)

resumo = pd.DataFrame({
    'dropout_% (valor=0)': {c: taxa_dropout(c).get(0, np.nan) for c in binarias},
    'dropout_% (valor=1)': {c: taxa_dropout(c).get(1, np.nan) for c in binarias},
}).round(1)
resumo['diferença'] = (resumo['dropout_% (valor=1)'] - resumo['dropout_% (valor=0)']).round(1)
resumo.sort_values('diferença', key=abs, ascending=False)

In [ ]:
# Proporção de cada classe dentro das binárias mais relevantes
relevantes = ['Tuition fees up to date', 'Debtor', 'Scholarship holder', 'Gender']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, relevantes):
    prop = (df.groupby(col)[TARGET].value_counts(normalize=True)
              .rename('p').reset_index())
    sns.barplot(data=prop, x=col, y='p', hue=TARGET, hue_order=['Dropout', 'Enrolled', 'Graduate'],
                palette=PALETTE, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('proporção')
    ax.legend([], frameon=False) if ax != axes[-1] else ax.legend(title='', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Análise bivariada — variáveis numéricas vs. evasão

As variáveis de **desempenho acadêmico** (disciplinas aprovadas e notas) são as que mais separam as classes — bem mais que dados demográficos ou socioeconômicos.

In [ ]:
comparar = ['Curricular units 1st sem (approved)', 'Curricular units 2nd sem (approved)',
            'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
            'Age at enrollment', 'Admission grade']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), comparar):
    sns.boxplot(data=df, x=TARGET, y=col, order=['Dropout', 'Enrolled', 'Graduate'],
                palette=PALETTE, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Médias por classe das variáveis de desempenho
df.groupby(TARGET)[comparar].mean().round(2).loc[['Dropout', 'Enrolled', 'Graduate']]

## 6. Correlação das variáveis com o alvo

Codificamos o alvo de forma ordenada por sucesso (`Dropout`=0 < `Enrolled`=1 < `Graduate`=2) e medimos a correlação de cada variável numérica. Serve como *ranking* de poder preditivo individual (linear).

In [ ]:
y_ord = df[TARGET].map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})
corr_alvo = df.select_dtypes('number').apply(lambda c: c.corr(y_ord)).drop(labels=[], errors='ignore')
corr_alvo = corr_alvo.sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
cores = ['#d62728' if v < 0 else '#2ca02c' for v in corr_alvo.values]
ax.barh(corr_alvo.index, corr_alvo.values, color=cores)
ax.set_title('Correlação com o alvo (Dropout < Enrolled < Graduate)')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()

## 7. Correlação entre features (redundância)

Os blocos do 1º e 2º semestre são quase espelhados (correlações ~0,90). Há forte redundância — relevante para modelos sensíveis a colinearidade e para possível redução de dimensionalidade.

In [ ]:
cols_corr = continuas + ['Application order']
cols_corr = list(dict.fromkeys(cols_corr))  # remove duplicatas mantendo ordem
corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Matriz de correlação — variáveis numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# Pares com correlação muito alta (|r| > 0.8)
alta = (corr.where(~mask).stack().rename('r').reset_index())
alta = alta[alta['r'].abs() > 0.8].sort_values('r', key=abs, ascending=False)
alta.round(3)

## 8. Síntese dos achados e implicações para a modelagem

**Qualidade dos dados**
- Sem valores nulos e sem duplicatas; nenhum tratamento de imputação necessário.
- Muitos inteiros são, na verdade, categorias nominais codificadas (alta cardinalidade, ex.: `Father's occupation` com 46 níveis) — não devem ser escalonados como números.

**Alvo**
- 3 classes desbalanceadas (Graduate 50% / Dropout 32% / Enrolled 18%). `Enrolled` é a minoria e tende a ser a mais difícil. → avaliar com **F1/recall macro**; considerar `class_weight='balanced'` ou reamostragem.

**Variáveis mais preditivas**
- Desempenho acadêmico domina: `Curricular units 2nd/1st sem (approved)` e `(grade)` têm as maiores correlações com o alvo (~0,49 a 0,62).
- `Tuition fees up to date` é fortíssima: **86,6%** de evasão entre quem **não** está em dia, contra **24,7%** entre quem está.
- `Scholarship holder` protege (12% de evasão vs 39%); `Debtor` agrava (62% vs 28%).
- Demográficas/macroeconômicas (`Nacionality`, ocupações dos pais, `Unemployment/Inflation/GDP`) têm correlação quase nula — candidatas a baixo valor preditivo.

**Redundância**
- Blocos do 1º e 2º semestre são quase idênticos (r ~ 0,90). Possível colinearidade → relevante para modelos lineares e para engenharia de atributos (ex.: agregar semestres).

**Próximos passos sugeridos**
1. Pré-processamento separando binárias / nominais (one-hot ou target encoding) / numéricas (scaler).
2. Baseline já disponível em `src/models/train.py`; comparar com modelos lineares e gradient boosting.
3. Avaliar importância de features para confirmar os achados e possivelmente descartar variáveis irrelevantes.